In [1]:
from mod1 import *

In [2]:
a = to_report()

In [ ]:
a.stock_select_with_Volume_Close(choice=2)

In [ ]:
a.get_graph(choice=2)

In [ ]:
import kkkk

In [ ]:
def stock_select_with_Volume_Close(choice = 2):

   if choice == 1:
       yesterday = input("어제날짜를 입력하세요 : sample: '2019-02-07'  ") or str_yesterday
       today = input("오늘날짜를 입력하세요 : sample: '2019-02-07'  ") or str_today
   
   else:
       day_df = pd.read_sql("select Date from market where Name='삼성전자' order by Date desc limit 2", engine)
       from_day = str(day_df['Date'][1])
       to_day = str(day_df['Date'][0])
       
   df  =select_stock('all', from_day, )
   df = df[df['Volume'] >  300000]
   df.reset_index(drop=True)
   #display(df)

   df1 = df[df['Date'].astype(str) == from_day]
   df1 = df1[['Name','Volume','Close']]
   df1.columns = ['Name','yester_Volume','yester_Close']
   #display(df1)


   df2 = df[df['Date'].astype(str) == to_day]
   df2 = df2[['Name','Volume','Close']]
   df2.columns = ['Name','today_Volume','today_Close']
   #display(df2)

   df3 = pd.merge(df1,df2,on='Name')
   df3['Close']=df3['today_Close']/df3['yester_Close']
   df3['Volume']=df3['today_Volume']/df3['yester_Volume']
   df3 = df3.sort_values(by=['Volume','Close'],ascending=False)
   df4 = df3.sort_values(by=['Close','Volume'],ascending=False)
   df3 = df3.reset_index(drop=True)

   df3 = df3[:15]
   df4 = df4.reset_index(drop=True)
   df4 = df4[:15]
   df3.to_excel(path_volume+str_today+'.xlsx')
   df4.to_excel(path_price+str_today+'.xlsx')        
   display(df3)
   display(df4) 
   return df3, df4      

df3, df4 = stock_select_with_Volume_Close(choice = 2)

In [17]:
df3.to_excel(f"{path_volume}{str_today}.xlsx")
df4.to_excel(f"{path_price}{str_today}.xlsx")


In [ ]:
path_volume

In [ ]:
def day_week_month_data(market='kospi', from_day = '2020-01-01', to_day = str_today, period ='month'):
    if market=='kospi' or market=='kosdaq':
        df = select_market_period(market,from_day)
    else :
        df = select_stock(market,from_day,to_day)
        
    df['Date']=pd.to_datetime(df['Date'])
    months = [g for n, g in df.groupby(pd.Grouper(key='Date',freq='M'))]  ##   월별
    weeks = [g for n, g in df.groupby(pd.Grouper(key='Date',freq='W'))]  ##   주별
    columns = ['Date','Open', 'High', 'Low', 'Close', 'Volume']
    rows = []    

    if period == 'day':
        nick = 'day'
        df=df[['Date','Open', 'High', 'Low','Close', 'Volume']]
        df.columns=columns
        #df = df.set_index(df['date'])
        #return df    
    if period == 'month':
        nick='month'
        period = months
    elif period == 'week':
        nick='week'
        period = weeks
        
    for i in range(len(period)):
        try:
            rows.append(period[i].iloc[-1]['Date'])
            rows.append(period[i].iloc[0]["Open"])
            rows.append(max(period[i]['High']))
            rows.append(min(period[i]['Low']))
            rows.append(period[i].iloc[-1]['Close'])
            rows.append(sum(period[i]['Volume']))
        except:
            pass
    
    #print('count:', len(period))
    arr = np.array(rows)
    per_week = len(arr)//6
    #per_month = len(arr)//6
    if nick == 'day':
        return df
    
    elif nick == 'week' :
        arr1 = arr.reshape(per_week,6)
    else:
        arr1 = arr.reshape(len(period),6)
        #arr1 = arr.reshape(per_month,6)
        
    df = pd.DataFrame(data=arr1, columns=columns)
    
    return df

In [ ]:
    to_day=str_today
    period = 'month'
    path_depress = 'C:/Users/linux/OneDrive/stockdata/depress/depress_'
    global from_day
    time_to_day = pd.to_datetime(to_day)
    
    if period == 'day':
        from_day = (time_to_day - timedelta(days=60)).strftime('%Y-%m-%d')
        from_day = str(from_day)

    elif period == 'week':
        from_day = (time_to_day - timedelta(days=180)).strftime('%Y-%m-%d')
        from_day = str(from_day)

    elif period == 'month':
        from_day = (time_to_day - timedelta(days=365*2)).strftime('%Y-%m-%d')
        from_day = str(from_day)
        print(from_day)